# 04 Cross-Platform (exploratory)

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['GE_DRIVE_ROOT'] = '/content/drive/MyDrive/hypothesaes_goemotions'
REPO_URL = 'https://github.com/YOUR_USERNAME/HypotheSAEs.git'   # <-- your fork
REPO_DIR = '/content/HypotheSAEs'

## Install

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
subprocess.run(['pip', 'install', '-q', '-e', REPO_DIR], check=True)
subprocess.run(['pip', 'install', '-q', '-U', 'datasets', 'sentence-transformers',
                'transformers', 'scikit-learn', 'statsmodels'], check=True)
# editable-install .pth files are only read at interpreter startup, so add the paths directly
import sys, importlib
for p in (REPO_DIR, os.path.join(REPO_DIR, 'goemotions')):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

## Imports

In [ ]:
import numpy as np
from scipy.stats import pearsonr
import ge_pipeline as ge
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.quickstart import train_sae
ge.set_seed()
SHARED = ['anger', 'fear', 'joy', 'sadness', 'surprise']   # Twitter has no disgust

## Load Corpora

In [ ]:
from datasets import load_dataset
ge_ds, NAMES = ge.load_goemotions()
ge_texts = list(ge_ds['train']['text'])
ge_all = ge.build_targets(ge_ds['train']['labels'], NAMES)
ge_targets = {k: ge_all[k] for k in SHARED}

tw = load_dataset('dair-ai/emotion', 'split')['train']
tw_names = tw.features['label'].names
tw_texts = list(tw['text'])
tw_lab = np.array(tw['label'])

def tw_target(name):
    keep = {tw_names.index('joy'), tw_names.index('love')} if name == 'joy' else {tw_names.index(name)}
    return np.array([1 if l in keep else 0 for l in tw_lab], dtype=np.int64)

tw_targets = {k: tw_target(k) for k in SHARED}
len(ge_texts), len(tw_texts)

## Embed and Pool

In [ ]:
e1 = get_local_embeddings(ge_texts, model=ge.EMBEDDER, nomic_prefix=ge.NOMIC_PREFIX,
                          cache_name='goemotions_modernbert', batch_size=128)
e2 = get_local_embeddings(tw_texts, model=ge.EMBEDDER, nomic_prefix=ge.NOMIC_PREFIX,
                          cache_name='twitter_modernbert', batch_size=128)
GE = np.stack([e1[t] for t in ge_texts]).astype(np.float32)
TW = np.stack([e2[t] for t in tw_texts]).astype(np.float32)
del e1, e2; ge.clear_mem()
pooled = np.concatenate([GE, TW], axis=0)
pooled.shape

## Shared SAE

In [ ]:
sae = train_sae(embeddings=pooled, M=1024, K=8,
                checkpoint_dir=os.path.join(ge.CKPT_DIR, 'crossplatform_M1024_K8'),
                n_epochs=100)
A_ge = sae.get_activations(GE)
A_tw = sae.get_activations(TW)
del pooled, GE, TW; ge.clear_mem()

## Per-Platform Predictiveness

In [ ]:
# in-sample: correlations use the SAE's own training data
def neuron_corrs(A, y):
    return np.nan_to_num(np.array([pearsonr(A[:, i], y)[0] for i in range(A.shape[1])]))

compare = {}
for cat in SHARED:
    cg = neuron_corrs(A_ge, ge_targets[cat])
    ct = neuron_corrs(A_tw, tw_targets[cat])
    top_ge = np.argsort(-np.abs(cg))[:10].tolist()
    top_tw = np.argsort(-np.abs(ct))[:10].tolist()
    compare[cat] = {
        'top_goemotions': [int(i) for i in top_ge],
        'top_twitter': [int(i) for i in top_tw],
        'overlap': int(len(set(top_ge) & set(top_tw))),
        'corr_of_neuron_corrs': float(pearsonr(cg, ct)[0]),
    }
ge.log_json('11_cross_platform', {'shared_labels': SHARED, 'comparison': compare})
ge.clear_mem()
compare